# Topic modeling with BERTopic

BERTopic est un modèle de sujet qui exploite les techniques de clustering et une variante de TF-IDF basée sur les classes pour la génération des sujets :
1. création des document embeddings à l'aide d'un modèle de langage pré-entraîné 
2. réduction de la dimensionnalité des embeddings des documents avant de créer des clusters de documents sémantiquement similaires, chacun représentant un sujet distinct
3. application d'une version de TF-IDF basée sur les classes afin d'extraire la représentation des sujets (*topic representation*) de chaque sujet.

*Topic* est un cluster de documents sémantiquement similaires
*Topic representation* est la façon dont les sujets sont décrits ou labélisés, ici par un set de mots extraits des documents du cluster.

=> https://medialab.sciencespo.fr/actu/les-enjeux-de-linformation-a-lere-numerique-vus-par-les-francais/

In [2]:
import polars as pl
from nltk.corpus import stopwords
# from spacy.lang.fr.stop_words import STOP_WORDS
# from spacy.lang.en.stop_words import STOP_WORDS
from spacy.lang.zh.stop_words import STOP_WORDS
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from bertopic.vectorizers import ClassTfidfTransformer
import sklearn
from sklearn.feature_extraction.text import CountVectorizer
from bertopic import BERTopic

In [ ]:
# nous avons besoin de `protobuf` version 3.20 pour `SentenceTransformer` avec camembert
# !pip install protobuf==3.20.0

# Préparation des données

Les modèles de sujet avec transformers et word-embedding ne demande, en théorie, aucun prétraitement des données. Cependant, il peut s'avérer utile de normaliser les données, d'enlever les balises HTML (quand ce n'est pas déjà fait), etc. Dans le cas de certains corpus, la suppression de mots spécifiques présents dans tout le corpus peut également s'avérer utile. Après cela, on peut transformer notre dataframe en list de documents.

In [3]:
input_file = "../data/danmaku.csv"

docs = pl.read_csv(input_file).with_columns(
    pl.col("text").str.normalize("NFC")
).filter(
    pl.col("text").is_not_null()
).get_column(
    "text"
).to_list() # remplacer captions par la colonne contenant vos publications
docs[0]

'看成皮眼子（错乱）'

In [4]:
len(docs)

14333

Après cela, nous préparons une liste de mots vides, que nous exclurons de la représentation des sujets.

In [5]:
stoplist = list(STOP_WORDS)
# ADDITIONAL_STOPWORDS = []
# stoplist.extend(ADDITIONAL_STOPWORDS)
stoplist[:5]

['但是', '千', '那样', '一定', '运用']

# Vectoriser les données

### Word embeddings

La première étape consiste à transformer une chaîne (phrase) en un tableau de nombres (vecteur), autrement dit à vectoriser le texte.
BERTopic peut convertir les documents en embeddings. Cependant, ce processus peut être très coûteux. Il est donc possible de calculer ces embeddings une seule fois et de les transmettre à BERTopic pour éviter de les calculer à chaque fois.

=> https://www.sbert.net/docs/sentence_transformer/pretrained_models.html
=> https://huggingface.co/spaces/mteb/leaderboard 

In [1]:
from transformers import AutoModel, AutoTokenizer

model_name = "sentence-transformers/distiluse-base-multilingual-cased-v1"
local_path = f"../model/{model_name}"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# tokenizer.save_pretrained(local_path)
# model.save_pretrained(local_path)

tokenizer_config.json:   0%|          | 0.00/452 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/556 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/539M [00:00<?, ?B/s]

In [9]:
sentences = ["真不知道这种主播是怎么火的", "游戏玩得菜还只会pua观众"]

embedding_model = SentenceTransformer(model_name) # ("all-MiniLM-L6-v2")
example_embeddings = embedding_model.encode(sentences, show_progress_bar=True)

print(f"Sentence 1 has {len(sentences[0])} characters, and the embedding is {len(example_embeddings[0])} long.")
print(f"Sentence 2 has {len(sentences[1])} characters, and the embedding is {len(example_embeddings[1])} long.")

modules.json:   0%|          | 0.00/341 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/2.24k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.58M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.58M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/114 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Sentence 1 has 13 characters, and the embedding is 512 long.
Sentence 2 has 13 characters, and the embedding is 512 long.


Comme le montre la cellule ci-dessus, les embeddings continues créées par le transformateur de phrases `dangvantuan/sentence-camembert-base` ont la même longueur, malgré des phrases de longueurs différentes. Ces embeddings continues ont la même longueur car, plutôt que de représenter directement les mots sous forme de nombres (c'est-à-dire de « sac de mots »), le transformateur crée une représentation riche prenant en compte 768 dimensions.

=> https://github.com/MaartenGr/BERTopic/discussions/822

### UMAP

BERTopic utilise généralement un algorithme de réduction de dimensionnalité (*dimensionality reduction algorithm*) pour réduire la taille des embeddings. Ceci permet d'éviter, dans une certaine mesure, le fléau de la dimension (https://en.wikipedia.org/wiki/Curse_of_dimensionality). 

UMAP est utilisé pour réduire l'espace dimensionnel. Cependant, il affiche par défaut un comportement stochastique qui produit des résultats différents à chaque exécution. Pour éviter cela pendant la période de test, nous pouvons définir un random_state du modèle avant de le transmettre à BERTopic.

```python
UMAP(angular_rp_forest=True, metric='cosine', n_components=10, n_neighbors=30, min_dist=0.1, random_state=42)
```

=> https://umap-learn.readthedocs.io/en/latest/parameters.html

### HDBSCAN

`nr_topics` est le paramètre qui permet de contrôler le nombre de sujets en fusionnant les sujets après leur création. Il permet de créer un nombre fixe de sujets. Il est toutefois conseillé de contrôler le nombre de sujets via le modèle de cluster, HDBSCAN par défaut. HDBSCAN possède un paramètre, `min_cluster_size`, qui contrôle indirectement le nombre de sujets créés. Un `min_cluster_size` élevé génère moins de sujets, tandis qu'un `min_cluster_size` faible en génère davantage.

```python
HDBSCAN(min_cluster_size=13, min_samples=3, prediction_data=True, metric='euclidean', cluster_selection_method='eom')
```

=> https://scikit-learn.org/stable/modules/generated/sklearn.cluster.HDBSCAN.html

### CountVectorizer

La représentation par défaut des sujets est calculée via c-TF-IDF. Cependant, c-TF-IDF s'appuie sur CountVectorizer, qui convertit le texte en unités. Grâce à CountVectorizer, nous pouvons effectuer plusieurs opérations : supprimer les mots vides, ignorer les mots rares, augmenter la range des n-gram. En d'autres termes, nous pouvons prétraiter les représentations des sujets après l'attribution des documents aux sujets. Cela n'aura aucune incidence sur le processus de clustering.

La tokenisation se fait automatiquement au niveau des espaces, pour le chinois il faut rajouter :

```python
import jieba

def tokenize_zh(text):
    words = jieba.lcut(text)
    return words

vectorizer = CountVectorizer(tokenizer=tokenize_zh)
```

### Construction du modèle

=> https://maartengr.github.io/BERTopic/getting_started/parameter%20tuning/parametertuning.html

In [10]:
import jieba

def tokenize_zh(text):
    words = jieba.lcut(text)
    return words

vectorizer = CountVectorizer(tokenizer=tokenize_zh)

In [11]:
def set_model_parameters(embedding_model=embedding_model, tokenizer=tokenize_zh):

    # Step 1 - extraction des embeddings
    # embedding_model = embedding_model

    # Step 2 - réduction des dimensionalité
    umap_model = UMAP(angular_rp_forest=True, metric='cosine', n_components=10, n_neighbors=30, min_dist=0.1)

    # Step 3 - Cluster reduced embeddings
    hdbscan_model = HDBSCAN(min_cluster_size=5, min_samples=5, prediction_data=True, metric='euclidean', cluster_selection_method='eom')
    # hdbscan_model = KMeans(n_clusters=50)
    # hdbscan_model = sklearn.cluster.AgglomerativeClustering(n_clusters=50)

    # Step 4 - Tokenisation des topics

    stoplist = list(STOP_WORDS)
    # ADDITIONAL_STOPWORDS = ["qu", "ya", "faut", "euh"]
    # stoplist.extend(ADDITIONAL_STOPWORDS)

    vectorizer_model = CountVectorizer(stop_words=stoplist, ngram_range=(1, 3), max_df=0.5, tokenizer=tokenizer)

    # Step 5 - Création de la représentation des topics
    ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True, bm25_weighting=True)

    # Topic model
    return BERTopic(
        embedding_model=embedding_model,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        ctfidf_model=ctfidf_model,
        language='chinese',
        n_gram_range=(1,3),
        nr_topics="auto"
        
    )

# Entrainement du modèle

On encode tous les documents de notre corpus.

In [12]:
# embedding_model = SentenceTransformer("dangvantuan/sentence-camembert-base")
embeddings = embedding_model.encode(docs, show_progress_bar=True)

Batches:   0%|          | 0/448 [00:00<?, ?it/s]

On crée une instance du modèle avec tous les paramètres définis et notre modèle d'embeddings.

In [13]:
topic_model = set_model_parameters(embedding_model)

On ajuste le modèle aux documents de notre corpus et aux embeddings que nous avons préparés.

In [14]:
topic_model.fit(docs, embeddings)

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
Building prefix dict from the default dictionary ...
Dumping model to file cache /var/folders/s9/j56343l57_s2lw7_47rw2bpr0000gn/T/jieba.cache
Loading model cost 0.363 seconds.
Prefix dict has been built successfully.


-1 est un "faux" cluster contenant toutes les valeurs aberrantes (*outliers*) ; il ne faut donc pas le prendre en compte. Un nom représentatif est attribué à chaque sujet. Le nom par défaut est un identifiant numérique suivi des mots-clés les plus représentatifs.

In [15]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,4293,-1_模拟_社会_时代_机飞,"[模拟, 社会, 时代, 机飞, 机飞 机飞, 时代 , 意识, 发展, 人工智能, 机飞...",[人工智能的终极形态是智人，完全是另外一种生物，和人工无关了。这还仅仅是人类可以理解的一种生...
1,0,448,0_ 声音_声音 _chatgpt_gpt 声音,"[ 声音, 声音 , chatgpt, gpt 声音, 配音, gpt4, 配音, ...","[你换个国内ai读搞不行吗，我听这香蕉口音真的听不清, 前面的红字 chatgpt是不用联..."
2,1,394,1_\ _\ _ 喔_喔,"[\ , \ , 喔, 喔 , 喔 , 喔, 喔, \, siu...","[喔！, 喔, @Cate Blanchett]"
3,2,255,2_逼 牛_ 牛 _牛 _ 牛 逼,"[逼 牛, 牛 , 牛 , 牛 逼, 牛 牛, 牛, 牛蛙, 牛...","[牛魔的我想听完这个故事其实，之前版本的编故事编的其实一般, 这个共享桌面牛了 相当于和全能..."
4,3,220,3_ 10 _10 _ 10_ 10,"[ 10 , 10 , 10, 10, 10 , 10 10, ...","[33？维斯塔潘！, 4o for all, 什么？10 个？]"
...,...,...,...,...,...
340,339,5,339_福音_福音 _快速 消除_患者 福音,"[福音, 福音 , 快速 消除, 患者 福音, 宅 男, 塌房 明星, 宅女 福音...","[宅男福音, 塌房明星快速消除术，真是福音呐, 帕金森患者的福音]"
341,340,5,340_源神_源神 启动_ 源神_启动,"[源神, 源神 启动, 源神, 启动, 启动 , 股神 启动 , 源神 启动, ...","[神识系统开启, 这下真是源神启动了, 源神 启动！]"
342,341,5,341_葛 大爷_葛_大爷_葛 大爷 哈哈哈,"[葛 大爷, 葛, 大爷, 葛 大爷 哈哈哈, 喷 点, 瘸 哈哈哈, 哈哈哈 大佬...","[哈哈，大佬也热衷于对喷, Ai爆发了文员完蛋了, 点进来还以为是葛大爷哈哈哈哈哈]"
343,342,5,342_算命 _算命_真的 能用 ai_真的 能用,"[算命 , 算命, 真的 能用 ai, 真的 能用, 终于 算命, 云 计算, 能...","[终于能算命了, 真的能用ai算命了, 意思是不是云计算？本地计算？]"


In [16]:
topic_model.get_topic(1)

[('\\    ', 0.24710525839535896),
 ('\\  ', 0.24710525839535896),
 ('    喔', 0.24710525839535896),
 ('喔  ', 0.24710525839535896),
 ('  喔  ', 0.24710525839535896),
 ('喔', 0.23107368016344865),
 ('  喔', 0.23107368016344865),
 ('\\', 0.21909953644730956),
 ('siuuuuuu    ', 0.1919399564231392),
 ('what    ', 0.1919399564231392)]

In [17]:
topic_model.generate_topic_labels(nr_words=20, topic_prefix=True, word_length=None, separator='  --  ')

['-1  --  模拟  --  社会  --  时代  --  机飞  --  机飞 机飞  --  时代    --  意识  --  发展  --  人工智能  --  机飞 机飞 机飞',
 '0  --    声音  --  声音    --  chatgpt  --  gpt 声音  --    配音  --  gpt4  --  配音  --    gpt 声音  --  gpt 声音    --  配音  ',
 '1  --  \\      --  \\    --      喔  --  喔    --    喔    --  喔  --    喔  --  \\  --  siuuuuuu      --  what    ',
 '2  --  逼   牛  --    牛    --  牛    --    牛 逼  --  牛   牛  --    牛  --    牛蛙  --  牛蛙    --    牛蛙    --  牛蛙',
 '3  --    10    --  10    --    10  --      10  --  10      --  10   10  --  10  --  12    --  12  --  00',
 '4  --  ai   乐子  --  乐子 ai    --  乐子 ai  --    乐子  --    乐子 ai  --  ai 画得  --    ai 画得  --  乐子  --  画得   ai  --  ai 画得  ',
 '5  --    开源  --  开源    --  开源   感谢  --  开源   开源  --    开源    --    感谢  --  感谢  --  开源  --    马斯克  --  马斯克',
 '6  --  视频    --  视频   视频  --    视频  --  生成 视频  --  做 视频  --  短 视频  --  视频 gpt  --  视频 ai 做  --  生 视频  --  短',
 '7  --    我超  --  我超  --      我超  --  天      --    我超    --  我超    --  天    --    天    --    天  --  布道  

In [ ]:
topic_model.save('last_bertopic.model')

# Exploration des prédictions du modèles

In [ ]:
topic_model = BERTopic.load('last_bertopic.model')

In [18]:
# topics_to_merge = [[3,4,6],[3,15]]
# topic_model.merge_topics(docs, topics_to_merge)

barchart = topic_model.visualize_barchart(top_n_topics=17, title="Représentation des topics", width=400, n_words=7)
barchart.write_html('barchart.html')

In [ ]:
topic_labels_dict = {
    0:"L'opinion & le journalisme",
    1:"Financement & l'indépendance des médias",
    2:"Désinformation"
}

topic_model.set_topic_labels(topic_labels_dict)
barchart = topic_model.visualize_barchart(top_n_topics=17, custom_labels=True, title="Représentation des topics<", width=800, n_words=7)
barchart.write_html('barchart.html')

In [19]:
hierarchical_topics = topic_model.hierarchical_topics(docs)
tree = topic_model.get_topic_tree(hierarchical_topics)
tree

100%|██████████| 343/343 [00:00<00:00, 445.51it/s]


'.\n├─厉害   厉害_  厉害  _  厉害_  太 厉害_太 厉害  \n│    ├─■──  太 厉害_太 厉害  _太 厉害_厉害   太_厉害   太大 ── Topic: 255\n│    └─厉害   厉害_  厉害  _  厉害_厉害  _厉害\n│         ├─■──厉害   太_  猛  _太强   猛_猛   厉害_厉害   好强 ── Topic: 251\n│         └─■──厉害   厉害_  厉害  _  厉害_厉害  _厉害 ── Topic: 205\n└─遥遥领先   遥遥领先_  声音_逼   牛_  赛博_奥特曼\n     ├─遥遥领先   遥遥领先_  声音_逼   牛_  赛博_奥特曼\n     │    ├─  nb  _nb   nb_  nb_nb  _nb\n     │    │    ├─■──nb   nb_  nb  _  nb_nb  _nb ── Topic: 73\n     │    │    └─■──  nb  _装修 nha_纽毕_nha   nb_nha   ── Topic: 306\n     │    └─遥遥领先   遥遥领先_  声音_逼   牛_  赛博_奥特曼\n     │         ├─遥遥领先   遥遥领先_  声音_逼   牛_  赛博_奥特曼\n     │         │    ├─机械 飞升_机械 飞升  _  机械 飞升_飞升   机械_飞升\n     │         │    │    ├─■──飞升   机械_  机械 飞升_机械 飞升_机械 飞升  _飞升 ── Topic: 100\n     │         │    │    └─■──机械 飞升  _机械 飞升_飞升  _瓢治能 机器_  1%   ── Topic: 312\n     │         │    └─遥遥领先   遥遥领先_  声音_逼   牛_  赛博_奥特曼\n     │         │         ├─遥遥领先   遥遥领先_  声音_逼   牛_  赛博_奥特曼\n     │         │         │    ├─和平_和平 和平_改 做_本质  _改\n     │         │     

In [20]:
hierarchical_topics = topic_model.hierarchical_topics(docs)
hierarchy = topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics, width=1200) # custom_labels=topic_labels_dict
hierarchy.write_html('hierarchy.html')

100%|██████████| 343/343 [00:00<00:00, 477.49it/s]


In [21]:
heatmap = topic_model.visualize_heatmap(n_clusters=3, width=1200) # custom_labels=topic_labels_dict
heatmap.write_html('heatmap.html')

Visualisation du classement de tous les termes sur tous les sujets. Chaque sujet est représenté par un ensemble de mots. Cependant, ces mots ne représentent pas tous le sujet de manière égale. Cette visualisation montre combien de mots sont nécessaires pour représenter un sujet et à partir de quel seuil l'effet bénéfique de l'ajout de mots commence à diminuer.

Il ya deux axes :
- x : Représente le rang des termes dans chaque sujet. Le terme avec le score c-TF-IDF le plus élevé a un rang de 1, le deuxième terme a un rang de 2, et ainsi de suite.
- y : Montre le score c-TF-IDF des termes. Plus le score est élevé, plus le terme est représentatif du sujet.

In [22]:
topic_model.visualize_term_rank(log_scale=True).write_html("term_rank.html")

In [23]:
topic_model.visualize_topics().write_html("visualize_topics.html")

Enregistrer les résultats dans un csv

In [24]:
PREDICTIONS_FILE = 'bertopic_topics_notebook.csv'

results = topic_model.get_document_info(docs=docs)
# results.to_csv()

# Latent Semantic Analysis (LSA)

LSA commence par créer une matrice terme-document. Les valeurs de la matrice sont généralement des scores TF-IDF qui pondèrent l'importance des mots dans chaque document. La matrice est ensuite décomposée en trois matrices à l'aide de la SVD. Ces matrices capturent les relations entre les termes et les documents. Enfin, en sélectionnant uniquement les premières composantes singulières, LSA réduit la dimensionalité tout en conservant les informations les plus importantes, ce qui permet de découvrir les sujets sous-jacents.

```python
embedding_model = sklearn.pipeline.make_pipeline(
    sklearn.feature_extraction.text.CountVectorizer(analyzer="word", tokenizer=lambda x: x.split(" ")),
    sklearn.decomposition.TruncatedSVD(10)
)
```

=> https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.TruncatedSVD.html#sklearn.decomposition.TruncatedSVD

In [26]:
def set_model_parameters_lsa():

    # Step 1 - extraction des embeddings
    embedding_model = sklearn.pipeline.make_pipeline(
        sklearn.feature_extraction.text.CountVectorizer(analyzer="word", tokenizer=tokenize_zh),
        sklearn.decomposition.TruncatedSVD(n_components=15)
    )

    # Step 2 - réduction des dimensionalité
    umap_model = UMAP(angular_rp_forest=True, metric='cosine', n_components=10, n_neighbors=20, min_dist=0.1)

    # Step 3 - Cluster reduced embeddings
    hdbscan_model = HDBSCAN(min_cluster_size=10, min_samples=5, prediction_data=True, metric='euclidean', cluster_selection_method='eom')
    # hdbscan_model = KMeans(n_clusters=50)
    # hdbscan_model = sklearn.cluster.AgglomerativeClustering(n_clusters=50)

    # Step 4 - Tokenisation des topics
    stoplist = list(STOP_WORDS)
    # ADDITIONAL_STOPWORDS = ["qu", "ya", "faut", "euh"]
    # stoplist.extend(ADDITIONAL_STOPWORDS)

    vectorizer_model = CountVectorizer(stop_words=stoplist, ngram_range=(1, 2))

    # Step 5 - Création de la représentation des topics
    ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True, bm25_weighting=True)

    # Topic model
    return BERTopic(
        embedding_model=embedding_model,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        ctfidf_model=ctfidf_model,
        language='chinese',
        n_gram_range=(1,2),
    )

In [27]:
topic_model = set_model_parameters_lsa()

In [28]:
topic_model.fit(docs)

In [29]:
topic_model.generate_topic_labels(nr_words=20, topic_prefix=True, word_length=None, separator='  --  ')

['-1  --  凑合  --  magi  --  magi magi  --  新效转换改做超越本质  --  凑合 凑合  --  穿越者  --  穿越者 穿越者  --  巧手先生  --  太牛了  --  很好',
 '0  --  生快 生快  --  协同 协同  --  协同  --  metaxr metaxr  --  生快  --  抢票 抢票  --  metaxr  --  抢票  --  牛蛙 牛蛙  --  八两半斤',
 '1  --  公平  --  公平 公平  --  底特律 变人  --  愚者抬杠  --  人工  --  别这样  --  4万人民币  --  罗老师  --  罗老师 别这样  --  extermination',
 '2  --  坐云上  --  假新闻  --  what  --  宇智波家族  --  清明节蹦迪  --  管泽元 se4  --  宇智波家族 管泽元  --  小白鼠  --  眼瞎  --  眯眯眼 what',
 '3  --  ai助手  --  ai助手 ai助手  --  ai就是未来  --  ai工具  --  反人类组织用ai搞破坏怎办 ai预测彩票呢  --  战斗力只有5 ai春晚  --  战斗力只有5  --  就是ai声线  --  怎么不选ai男士 悄悄投入一批ai人到社会中  --  悄悄投入一批ai人到社会中 什么时候开始卖ai伴侣',
 '4  --  太正确了  --  形象还是一个黑人 太正确了  --  形象还是一个黑人  --  没有啊  --  落后太多了  --  有些畸变 但是太牛了  --  机翻成纳米了 物竞天择  --  水流向反了 人在飘  --  机械飞升 又又在大吹特吹了  --  0差不多',
 '5  --  夺少  --  科幻  --  还有高手  --  纳米级  --  没有本质不同  --  等会儿可持续  --  纯记忆功能  --  算了吧 艾达  --  科幻 一帆翻译  --  红队 啥意思',
 '6  --  五彩斑斓的黑  --  五彩斑斓的黑 五彩斑斓的黑  --  强者的发型  --  四条腿各走各的  --  智慧的眼神  --  矫健的步伐 集体摸鱼的厨师  --  矫健的步伐

In [30]:
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,1908,-1_凑合_magi_magi magi_新效转换改做超越本质,"[凑合, magi, magi magi, 新效转换改做超越本质, 凑合 凑合, 穿越者, ...",[我觉得不用三年，估计很多视频软件，剪辑软件，PS工具，文书软件，甚至写代码的工具，都可以滚...
1,0,497,0_生快 生快_协同 协同_协同_metaxr metaxr,"[生快 生快, 协同 协同, 协同, metaxr metaxr, 生快, 抢票 抢票, m...","[协同, 协同, 协同]"
2,1,339,1_公平_公平 公平_底特律 变人_愚者抬杠,"[公平, 公平 公平, 底特律 变人, 愚者抬杠, 人工, 别这样, 4万人民币, 罗老师,...","[奈何本人没文化，一句我操走天下，以能为人作我师，谦虚下功学知识。机器，机器呀, 注定不能完..."
3,2,220,2_坐云上_假新闻_what_宇智波家族,"[坐云上, 假新闻, what, 宇智波家族, 清明节蹦迪, 管泽元 se4, 宇智波家族 ...","[假新闻？, 坐云上？, 坐云上？]"
4,3,159,3_ai助手_ai助手 ai助手_ai就是未来_ai工具,"[ai助手, ai助手 ai助手, ai就是未来, ai工具, 反人类组织用ai搞破坏怎办 ...","[现在AI只是拿资料去训练它...逻辑工程, AI工具：GPT-SoVITS, AI开发游戏..."
...,...,...,...,...,...
426,425,10,425_程序员这不是有工作了_程序员这不是有工作了 这就别拿出来秀了罢_这我们不得上红毯礼仪...,"[程序员这不是有工作了, 程序员这不是有工作了 这就别拿出来秀了罢, 这我们不得上红毯礼仪小...","[这勺子跟变魔术一样就出来了, 这东西是不是该驱动就可以了, 这就别放出来了]"
427,426,10,426_卷的觉都睡不好_卷的觉都睡不好 他们都不接马斯克的梗哈哈_阿瓦隆系统这类的每个平台都...,"[卷的觉都睡不好, 卷的觉都睡不好 他们都不接马斯克的梗哈哈, 阿瓦隆系统这类的每个平台都有...","[阿瓦隆系统这类的每个平台都有, 用过都知道-4输出有点慢的, 潜台词：每个国家都要采购我们..."
428,427,10,427_美帝核心科技_文明才能掌握科技_科技属于文明_核心科技 美帝核心科技,"[美帝核心科技, 文明才能掌握科技, 科技属于文明, 核心科技 美帝核心科技, 感谢科技 美...","[科技垃圾, 美帝核心科技, 美帝核心科技]"
429,428,10,428_所以sora搞出来没有_玩呢_没男人_没化妆 任何事情,"[所以sora搞出来没有, 玩呢, 没男人, 没化妆 任何事情, 没化妆, 显的有点变态了 ...","[失业？新就业方向, 玩呢？相差这么多, 任何事情？变个卡车doge]"
